In [1]:
import pandas as pd
import numpy as np
import plotly.graph_objects as go
import plotly.io as pio
from plotly.subplots import make_subplots
import os

# Set renderer to notebook
pio.renderers.default = "notebook"

# Fix for MathJax deprecation warning
pio.defaults.mathjax = None

In [2]:
# 1. Load Data
# Ensure this path matches your file location
df = pd.read_csv('../data/FullJoin3_with_climate_ratings-proofed.csv')

# 2. Rename columns
rename_dict = {
    'climate_rating_current_sheet': 'Current',
    'climate_rating_emissions_limited_2050': '2050',
    'climate_rating_business_as_usual_2090': '2090',
    'climate_rating_bau_plus_1degree_2090': '2090_plus_1_degree',
    'GenusSpecies': 'Taxon',
}

# 3. Standardize Coordinates & Filter
df['LocationCoordY_fixed'] = df[['LocationCoordX', 'LocationCoordY']].min(axis=1) # Longitude
df['LocationCoordX_fixed'] = df[['LocationCoordX', 'LocationCoordY']].max(axis=1) # Latitude

exclude_locs = ['Nursery', 'Nitobe Memorial Garden', 'Nitobe']
df_clean = df[~df['LocationName'].isin(exclude_locs)].copy()
df_clean = df_clean[df_clean['LocationCoordY_fixed'] < -123.245].copy()

# 4. Apply Jitter
jitter_strength = 0.00006 
np.random.seed(42) 
df_clean['LocationCoordX_Jittered'] = df_clean['LocationCoordX_fixed'] + np.random.uniform(-jitter_strength, jitter_strength, size=len(df_clean))
df_clean['LocationCoordY_Jittered'] = df_clean['LocationCoordY_fixed'] + np.random.uniform(-jitter_strength, jitter_strength, size=len(df_clean))

# 5. Reshape Data
df_ready = df_clean.rename(columns=rename_dict)
cols_to_keep = [
    'ItemAccNoFull', 'LocationCoordX_Jittered', 'LocationCoordY_Jittered', 
    'LifeForm', 'Taxon', 
    'Current', '2050', '2090', '2090_plus_1_degree'
]
final_merged_df = df_ready[cols_to_keep].copy()

lifeform_mapping = {
    'Shrub': 'Woody', 'Tree': 'Woody', 'Shrub or Tree': 'Woody',
    'Climber_Liana_Vine': 'Woody', 'Herbaceous Perennial': 'Perennial',
    'Bulb, Corm, or Tuber': 'Perennial', 'Annual': 'Short-lived',
    'Biennial': 'Short-lived', 'Habit unknown': 'Unknown'
}
final_merged_df['LifeForm'] = final_merged_df['LifeForm'].map(lifeform_mapping).fillna('Unknown')

long_format_df = pd.melt(
    final_merged_df,
    id_vars=['ItemAccNoFull', 'LocationCoordX_Jittered', 'LocationCoordY_Jittered', 'Taxon', 'LifeForm'],
    value_vars=['Current', '2050', '2090', '2090_plus_1_degree'],
    var_name='Era',
    value_name='ClimateRating'
).dropna(subset=['ClimateRating'])

# 6. Filter for Plotting
target_lifeforms = ['Woody', 'Perennial']
plot_df = long_format_df[long_format_df['LifeForm'].isin(target_lifeforms)].copy()

print(f"Data Ready: {len(plot_df)} points.")

Data Ready: 47904 points.


In [4]:
# --- Interactive Layer Map ---
# This creates the single map with toggleable layers

fig_interactive = go.Figure()

# Define order
eras = ['Current', '2050', '2090', '2090_plus_1_degree']
lifeforms = ['Woody', 'Perennial']

# Loop to create 8 separate traces (Layers)
for era in eras:
    for lf in lifeforms:
        subset = plot_df[(plot_df['Era'] == era) & (plot_df['LifeForm'] == lf)]
        
        fig_interactive.add_trace(go.Scattermap(
            lat=subset['LocationCoordX_Jittered'],
            lon=subset['LocationCoordY_Jittered'],
            mode='markers',
            marker=dict(
                size=6,
                color=subset['ClimateRating'],
                colorscale=['red', 'yellow', 'green'], # Red=0 (Risk), Green=10 (Safe)
                cmin=0, cmax=11,
                opacity=0.8
            ),
            name=f"{era} - {lf}", # This name appears in the legend
            text=subset['Taxon'],
            hoverinfo='text+name'
        ))

# Configure Layout
center_lat = plot_df['LocationCoordX_Jittered'].mean()
center_lon = plot_df['LocationCoordY_Jittered'].mean()

fig_interactive.update_layout(
    title="UBCBG Climate Risk Layers (Click Legend to Toggle)",
    map=dict(
        style="open-street-map",
        center=dict(lat=center_lat, lon=center_lon),
        zoom=16
    ),
    legend_title_text="Layer (Era - LifeForm)",
    height=800,
    margin={"r":0,"t":40,"l":0,"b":0}
)

fig_interactive.show()

In [7]:
# Dictionary to store the generated figures
figures = {}

# Layout: 2x2 Matrix (Current, 2050, 2090, 2090+1)
era_matrix = [
    ['Current', '2050'],
    ['2090', '2090_plus_1_degree']
]

for lf in target_lifeforms:
    # Create 2x2 Subplot Grid
    fig = make_subplots(
        rows=2, cols=2,
        subplot_titles=[f"{lf} - Current", f"{lf} - 2050", f"{lf} - 2090", f"{lf} - 2090 (+1°)"],
        specs=[[{"type": "map"}, {"type": "map"}], [{"type": "map"}, {"type": "map"}]],
        vertical_spacing=0.08, horizontal_spacing=0.04
    )

    # Fill Grid
    for r in range(2):
        for c in range(2):
            era = era_matrix[r][c]
            subset = plot_df[(plot_df['Era'] == era) & (plot_df['LifeForm'] == lf)]
            
            # Show legend only on top-right plot
            show_legend = (r == 0 and c == 1)
            
            fig.add_trace(go.Scattermap(
                lat=subset['LocationCoordX_Jittered'],
                lon=subset['LocationCoordY_Jittered'],
                mode='markers',
                marker=dict(
                    size=6, color=subset['ClimateRating'],
                    colorscale=['red', 'yellow', 'green'], cmin=0, cmax=11,
                    opacity=0.8, showscale=show_legend,
                    colorbar=dict(title="Rating", len=0.5, y=0.8) if show_legend else None
                ),
                name=era, text=subset['Taxon'], hoverinfo='text+name'
            ), row=r+1, col=c+1)

    # Styling
    center_lat = plot_df['LocationCoordX_Jittered'].mean()
    center_lon = plot_df['LocationCoordY_Jittered'].mean()
    common_map_style = dict(style="open-street-map", center=dict(lat=center_lat, lon=center_lon), zoom=15)
    
    # Apply style to all 4 subplots
    for i in range(1, 5):
        fig.update_layout({f'map{"" if i==1 else i}': common_map_style})
    
    fig.update_layout(
        height=1000, width=1000,
        title_text=f"Climate Risk: {lf} Plants (2x2 Matrix)",
        margin={"r":20,"t":80,"l":20,"b":20}, showlegend=False
    )
    
    # Store figure and show
    figures[lf] = fig
    fig.show()

In [6]:
# --- High-Resolution Export Tool ---
from datetime import datetime

def export_map_png(fig, filename=None, width=2400, height=1400, scale=3):
    """
    Exports a specific Plotly figure to PNG.
    """
    if fig is None:
        raise ValueError("No figure provided. Pass a figure object to this function.")

    if filename is None:
        ts = datetime.now().strftime('%Y%m%d_%H%M%S')
        filename = f"UBCBG_map_export_{ts}.png"

    # Export
    fig.write_image(filename, width=width, height=height, scale=scale)
    print(f"Exported: {filename} (width={width}, height={height}, scale={scale})")

# --- USAGE EXAMPLES ---

# 1. Export the Interactive Map (whichever layers are currently visible)
# export_map_png(fig_interactive, filename="UBCBG_Interactive_Layers.png")

# 2. Export the Woody 2x2 Grid
# export_map_png(figures['Woody'], filename="UBCBG_Woody_Grid.png", scale=3)

# 3. Export the Perennial 2x2 Grid
#export_map_png(figures['Perennial'], filename="UBCBG_Perennial_Grid.png", scale=3)

Exported: UBCBG_Perennial_Grid.png (width=2400, height=1400, scale=3)
